In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('fintech360.bronze.bank_data')
display(df)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType,
    DecimalType, BooleanType, LongType
)

BANK_PARENT_SCHEMA = StructType([
    StructField("bank_txn_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("account_id", StringType(), True),
    StructField("bank_name", StringType(), True),
    StructField("transaction_ts", TimestampType(), True),
    StructField("transaction_type", StringType(), True),
    StructField("transaction_amount", DecimalType(18, 2), True),
    StructField("signed_amount", DecimalType(18, 2), True),
    StructField("balance_after_txn", DecimalType(18, 2), True),
    StructField("currency", StringType(), True),
    StructField("narration", StringType(), True),
    StructField("transaction_category", StringType(), True),
    StructField("is_salary_like", BooleanType(), True),
    StructField("source_file_id", StringType(), True),
    StructField("created_at", LongType(), True),
    StructField("updated_at", LongType(), True),
    StructField("bank_details_json", StringType(), True),
    StructField("_rescued_data", StringType(), True),
    StructField("_ingested_at", TimestampType(), True),
    StructField("_source_file", StringType(), True)
])

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType, StringType

def normalize_bank_schema(df, parent_schema=BANK_PARENT_SCHEMA):
    parent_fields = {field.name: field for field in parent_schema.fields}
    parent_names = list(parent_fields.keys())
    incoming_names = df.columns

    # Identify unexpected source columns before modifying the DataFrame
    extra_cols = [c for c in incoming_names if c not in parent_names]

    # Preserve unexpected columns for investigation
    if extra_cols:
        extra_json = F.to_json(
            F.struct(*[F.col(c) for c in extra_cols])
        )
        if "_rescued_data" in incoming_names:
            df = df.withColumn(
                "_rescued_data",
                F.when(
                    F.col("_rescued_data").isNull(),
                    extra_json
                ).otherwise(
                    F.concat_ws(
                        "\n",
                        F.col("_rescued_data"),
                        extra_json
                    )
                )
            )
        else:
            df = df.withColumn("_rescued_data", extra_json)

    # Normalize customer IDs before casting to the parent type
    if "customer_id" in df.columns:
        customer_id_text = F.trim(F.col("customer_id").cast("string"))
        customer_id_number = customer_id_text.cast("decimal(38,10)")

        df = df.withColumn(
            "customer_id",
            F.when(
                customer_id_text.rlike(r"^[+-]?\d+\.0+$"),
                customer_id_number.cast("decimal(38,0)").cast("string")
            ).otherwise(customer_id_text)
        )

    # Add missing columns and cast all parent columns
    for name, field in parent_fields.items():
        if name not in df.columns:
            df = df.withColumn(
                name,
                F.lit(None).cast(field.dataType)
            )
        else:
            df = df.withColumn(
                name,
                F.col(name).cast(field.dataType)
            )

    # Select exactly the parent columns in the required order
    return df.select(*parent_names)

In [0]:
df = normalize_bank_schema(df)

df.printSchema()
display(df.limit(10))

In [0]:
(df.write
    .mode('append')
    .format('delta')
    .saveAsTable('fintech360.schema_validation.bank_data'))